In [1]:
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
import matplotlib.pyplot as plt

# Trying the 1D-CNN of the smallest sensor dataframe (Heat exchanger)

In [2]:
save_directory = './preprocessed_dataset/VG5/op_0'

control_df = pd.read_parquet(path=f'{save_directory}/control.parquet')
sensors_heat_exchanger = pd.read_parquet(path=f'{save_directory}/sensors_heat_exchanger.parquet')

training_df = pd.concat([control_df,sensors_heat_exchanger], axis=1)

training_df.shape

(99218, 16)

In [3]:
control_df.shape

(99218, 7)

### Snippets of code

In [ ]:
import torch.nn as nn

def init_weights(m):
    if isinstance(m, nn.BatchNorm1d):
        m.weight.data.fill_(1.0)
        m.bias.data.zero_()
    elif isinstance(m, nn.Conv1d) or isinstance(m, nn.Linear):
        m.weight.data = nn.init.xavier_uniform_(
            m.weight.data, gain=nn.init.calculate_gain('relu'))
        if m.bias is not None:
            m.bias.data.zero_()

class CNN(nn.Module):
        
    """
    Args:
        n_features (int, optional): number of input features. Defaults to 18.
        window (int, optional): sequence length. Defaults to 50.
        n_ch (int, optional): number of channels (filter size). Defaults to 10.
        n_k (int, optional): kernel size. Defaults to 10.
        n_hidden (int, optional): number of hidden neurons for regressor. Defaults to 50.
        n_layers (int, optional): number of convolution layers. Defaults to 5.
    """
    
    def __init__(self, 
                 in_channels=18, 
                 out_channels=1,
                 window=50, 
                 n_ch=20, 
                 n_k=10, 
                 n_hidden=50, 
                 n_layers=3,
                 dropout=0.0,
                 padding='same',
                 use_batchnorm=False):
        super().__init__()
        
        # Create a ModuleList to hold variable number of conv layers
        self.conv_layers = nn.ModuleList()
        
        # First layer (input layer)
        self.conv_layers.append(nn.Sequential(
            nn.Conv1d(in_channels, n_ch, kernel_size=n_k, padding=padding),
            nn.BatchNorm1d(n_ch) if use_batchnorm else nn.Identity(),
            nn.ReLU(),
            nn.Dropout(dropout)
        ))
        
        for _ in range(n_layers - 2):  
            self.conv_layers.append(nn.Sequential(
                nn.Conv1d(n_ch, n_ch, kernel_size=n_k, padding=padding),
                nn.BatchNorm1d(n_ch) if use_batchnorm else nn.Identity(),
                nn.ReLU(),
                nn.Dropout(dropout)
            ))

        self.conv_layers.append(nn.Sequential(
            nn.Conv1d(n_ch, 1, kernel_size=n_k, padding=padding),
            nn.BatchNorm1d(1) if use_batchnorm else nn.Identity(),
            nn.ReLU(),
            nn.Dropout(dropout)
        ))
        
        flat_features = window  

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_features, n_hidden),
            nn.BatchNorm1d(n_hidden) if use_batchnorm else nn.Identity(),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(n_hidden, out_channels)  # Output layer
        )
        
        # Initialize weights
        self.apply(init_weights)
        
    def forward(self, x):
        # Pass through all conv layers sequentially
        for layer in self.conv_layers:
            x = layer(x)
        x = self.regressor(x)
        return x
    

# Input:          [256, 18, 50]  # (batch, features, window)
# After layer1:   [256, 10, 50]  # (batch, channels, window)
# After layer2:   [256, 10, 50]  # (batch, channels, window)
# After layer3:   [256, 1, 50]   # (batch, channels, window)
# After flatten:  [256, 50]      # (batch, flattened)
# Final output:   [256, 1]       # (batch, prediction)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 10, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(10)  ###Add batch norm
        self.conv2 = nn.Conv1d(10, 10, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(10)  ###Add batch norm
        self.conv3 = nn.Conv1d(10, 10, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(10)  ###Add batch norm
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(10 * 512, 256)  # 512 is input dimension

    def forward(self, x):
        x = self.dropout(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc(x))
        return x


class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(256, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


class BaselineModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = FeatureExtractor()
        self.classifier = Classifier()

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.classifier(features)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import linalg as LA
import torch.optim as optim
import numpy as np
from tqdm import tqdm

def train_baseline(model, source_loader, target_loader, args, device):
    """Standard source training"""
    print("\nTraining Baseline Model...")

    # Included this to save the metric
    with open("results/baseline_metrics.txt", "w") as f:
        f.write("")

    optimizer = optim.Adam(model.parameters(), lr=args.lr)

    for epoch in range(args.epochs):
        model.train()
        total_loss = 0

        for data, target in tqdm(source_loader, desc=f"Epoch {epoch}"):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Calculate training and testing metrics
        source_loss, source_acc = evaluate(model, source_loader, device)
        target_loss, target_acc = evaluate(model, target_loader, device)

        # Save metrics for plotting
        save_metrics("baseline", epoch, source_loss, source_acc, target_acc)

        # Print loss and accuracy for source and target
        print(f"Training loss : {source_loss} - accuracy : {source_acc}")
        print(f"Test loss : {target_loss} - accuracy : {target_acc}")

    # Save final model
    torch.save(model.state_dict(), "final_baseline.pth")

    # Return Final target accuracy
    return target_acc


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_loss += F.nll_loss(output, target).item()
            pred = output.max(1)[1]
            correct += pred.eq(target).sum().item()
            total += target.size(0)

    return total_loss / len(loader), correct / total


# Save metric to txt file for backup
def save_metrics(method_name, epoch, source_loss, source_acc, target_acc):
    with open(f"results/{method_name}_metrics.txt", "a") as f:
        f.write(f"{epoch},{source_loss},{source_acc},{target_acc}\n")

### Example of a 1D-CNN by chatgpt

Output Layer:

Use a Dense layer with 1 neuron and a sigmoid activation function for binary classification.

In [ ]:
# Define the input shape
input_shape = (100000, 16)

# Build the model
model = Sequential()

# Add convolutional layers
model.add(Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

# Flatten the output from the convolutional layers
model.add(Flatten())

# Add dense layers to learn higher-level features
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))  # Dropout layer to prevent overfitting

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))

# Output layer for binary classification
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()